# DP2 — DiaObject Light Curves (DiaSource + ForcedPhotometry) — COSMOS DDF

**Author:** dagoret  
**Creation Date:** 2026-03-14  
**Last Date:** 2026-03-14  
**Version:** v3

## Purpose

Read the 3 CSV files generated by `2026-03-13_DP2_DDF_DiaObjects_Butler.ipynb`:
- `DiaObjects_COSMOS_nmin200.csv`
- `DiaSources_COSMOS_nmin200.csv`
- `ForcedSrc_COSMOS_nmin200.csv`

Then plot **multi-band light curves** (u, g, r, i, z, y) for a subset of DiaObjects  
selected by `nDiaSources > CUT_NDIASOURCES`.

Each figure shows one DiaObject with:
- **Marker `o`** : DiaSource detections (psf flux or AB magnitude)
- **Marker `+`** : Forced photometry
- **Dashed lines (`--`)** connecting points in MJD order within each band
- **Error bars** for both source types
- **Independent y-axes** (twin axes) per band, offset vertically
- **Fixed y-axis range** : 16–28 mag  (or equivalent flux range in nJy)
- **Top x-axis** : Calendar date (YYYY-MM-DD)
- **Bottom x-axis** : MJD
- **Interactive zoom/pan widgets** via `%matplotlib widget` (ipympl)

## Input data
Generated by: `2026-03-13_DP2_DDF_DiaObjects_Butler.ipynb`  
Data folder: `data_DP2_DDF_DIAOBJ_BUTLER_01/`

## Requirements
```
pip install ipympl
```

---
## 0. Interactive backend — ipympl (widgets zoom/pan)

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

---
## 1. Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from astropy.time import Time

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
mpl.rcParams.update(
    {
        "font.size": 13,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 11,
        "legend.fontsize": 12,
        "legend.title_fontsize": 13,
        "figure.titlesize": 18,
    }
)

---
## 2. Parameters

In [ ]:
# ─── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = "data_DP2_DDF_DIAOBJ_BUTLER_01"

FILE_DIAOBJ = f"{DATA_DIR}/DiaObjects_COSMOS_nmin200.csv"
FILE_DIASRC = f"{DATA_DIR}/DiaSources_COSMOS_nmin200.csv"
FILE_FORCEDSRC = f"{DATA_DIR}/ForcedSrc_COSMOS_nmin200.csv"

# ─── Selection ────────────────────────────────────────────────────────────────
# Minimum number of DiaSource detections to select a DiaObject
CUT_NDIASOURCES = 250  # select objects with nDiaSources > this value

# Maximum number of light curves to display (to avoid overloading the notebook)
MAX_CURVES = 50  # set to None to plot all selected objects

# ─── Plot mode ────────────────────────────────────────────────────────────────
# 'flux'  → display psfFlux in nJy
# 'mag'   → display AB magnitude  (-2.5*log10(flux / 3631e9) for flux in nJy)
PLOT_MODE = "mag"  # 'flux' or 'mag'

# ─── Figure geometry ──────────────────────────────────────────────────────────
# Wide figure to stretch the time axis; height kept moderate for readability
FIG_WIDTH = 16  # inches  — increase to stretch x-axis further
FIG_HEIGHT = 5.0  # inches

# Number of date ticks on the top axis
N_DATE_TICKS = 8

# ─── Line style for the dashed connector between points ───────────────────────
LINE_STYLE = "--"  # dashed line connecting points in MJD order
LINE_ALPHA = 0.45  # transparency of the connecting line
LINE_WIDTH = 0.9

# ─── Bands & cosmetics ────────────────────────────────────────────────────────
BANDS = ["u", "g", "r", "i", "z", "y"]
COLORS = {"u": "#7b2d8b", "g": "#1f8b4c", "r": "#e74c3c", "i": "#e67e22", "z": "#2980b9", "y": "#8e44ad"}
# Spine offset index: band 'u' has the outermost (largest) offset
OFFSETS = {"u": 5, "g": 4, "r": 3, "i": 2, "z": 1, "y": 0}
SPINE_STEP_PX = 65  # pixels between successive y-axis spines

# ─── Y-axis limits ────────────────────────────────────────────────────────────
# Magnitude mode: MAG_MIN (bright end) to MAG_MAX (faint end)
# The y-axis is inverted, so MAG_MIN appears at the TOP of the plot.
MAG_MIN = 16.0  # brightest displayed AB magnitude
MAG_MAX = 28.0  # faintest  displayed AB magnitude

# Flux mode: equivalent limits derived from the magnitude limits.
# m_AB = -2.5*log10(F / ZP_FLUX_NJY)  →  F = ZP_FLUX_NJY * 10^(-m/2.5)  [nJy]
# MAG_MIN=16 → high flux (bright), MAG_MAX=28 → low flux (faint)
# These are computed after ZP_FLUX_NJY is defined (see below).

# ─── Flux → magnitude conversion ──────────────────────────────────────────────
# LSST zero-point: m_AB = -2.5 * log10(flux_nJy / 3631e9)
ZP_FLUX_NJY = 3631e9  # nJy at mag = 0

# Flux limits equivalent to MAG_MIN and MAG_MAX
FLUX_MAX = ZP_FLUX_NJY * 10 ** (-MAG_MIN / 2.5)  # nJy at MAG_MIN=16 (bright → high flux)
FLUX_MIN = ZP_FLUX_NJY * 10 ** (-MAG_MAX / 2.5)  # nJy at MAG_MAX=28 (faint  → low  flux)


def flux_to_mag(flux_nJy):
    """Convert flux (nJy) to AB magnitude; returns NaN for non-positive flux."""
    with np.errstate(divide="ignore", invalid="ignore"):
        mag = np.where(
            flux_nJy > 0, -2.5 * np.log10(np.where(flux_nJy > 0, flux_nJy, np.nan) / ZP_FLUX_NJY), np.nan
        )
    return mag


def flux_err_to_mag_err(flux_nJy, flux_err_nJy):
    """Propagate flux error to magnitude error (linear approximation)."""
    with np.errstate(divide="ignore", invalid="ignore"):
        mag_err = np.where(
            flux_nJy > 0,
            (2.5 / np.log(10)) * np.abs(flux_err_nJy / np.where(flux_nJy > 0, flux_nJy, np.nan)),
            np.nan,
        )
    return mag_err


print(f"CUT_NDIASOURCES : {CUT_NDIASOURCES}")
print(f"MAX_CURVES      : {MAX_CURVES}")
print(f"PLOT_MODE       : {PLOT_MODE}")
print(f"Figure size     : {FIG_WIDTH} × {FIG_HEIGHT} in")
print(f"Y-axis mag      : [{MAG_MIN}, {MAG_MAX}]  (bright → faint)")
print(f"Y-axis flux     : [{FLUX_MIN:.3e}, {FLUX_MAX:.3e}] nJy  (faint → bright)")

---
## 3. Load data

In [ ]:
df_obj = pd.read_csv(FILE_DIAOBJ)
df_src = pd.read_csv(FILE_DIASRC)
df_frc = pd.read_csv(FILE_FORCEDSRC)

print(f"DiaObjects  : {len(df_obj):,} rows,  columns: {list(df_obj.columns[:6])} ...")
print(f"DiaSources  : {len(df_src):,} rows,  columns: {list(df_src.columns[:6])} ...")
print(f"ForcedSrc   : {len(df_frc):,} rows,  columns: {list(df_frc.columns[:6])} ...")

In [ ]:
print(df_obj["nDiaSources"].describe())
print(
    f"\nObjects with nDiaSources > {CUT_NDIASOURCES}: " f"{(df_obj['nDiaSources'] > CUT_NDIASOURCES).sum()}"
)

---
## 4. Select DiaObjects

In [ ]:
mask_cut = df_obj["nDiaSources"] > CUT_NDIASOURCES
df_obj_sel = df_obj[mask_cut].sort_values("nDiaSources", ascending=False).reset_index(drop=True)

print(f"Selected DiaObjects (nDiaSources > {CUT_NDIASOURCES}): {len(df_obj_sel)}")

df_obj_plot = df_obj_sel.head(MAX_CURVES) if MAX_CURVES is not None else df_obj_sel
print(f"Will plot {len(df_obj_plot)} light curves  (MAX_CURVES={MAX_CURVES})")
display(df_obj_plot[["diaObjectId", "ra", "dec", "nDiaSources"]].head(20))

---
## 5. Helpers

In [ ]:
def make_date_axis(ax_mjd, mjd_min, mjd_max, n_ticks=8):
    """
    Add a secondary x-axis on top of `ax_mjd` showing calendar dates.

    The secondary axis shares the same data-space limits as `ax_mjd`.
    Tick positions are evenly spaced in MJD; labels are formatted as
    YYYY-MM-DD using astropy.time.Time.

    Parameters
    ----------
    ax_mjd : matplotlib.axes.Axes
    mjd_min, mjd_max : float   — limits of the MJD axis
    n_ticks : int              — number of date ticks

    Returns
    -------
    ax_date : matplotlib.axes.Axes  (the new top axis)
    """
    ax_date = ax_mjd.twiny()
    ax_date.set_xlim(ax_mjd.get_xlim())

    tick_mjds = np.linspace(mjd_min, mjd_max, n_ticks)
    tick_labels = [Time(m, format="mjd", scale="tai").strftime("%Y-%m-%d") for m in tick_mjds]

    ax_date.set_xticks(tick_mjds)
    ax_date.set_xticklabels(tick_labels, rotation=30, ha="left", fontsize=10)
    ax_date.xaxis.set_label_position("top")
    ax_date.set_xlabel("Date (UTC)", fontsize=11, labelpad=8)
    return ax_date


def _plot_band(ax_b, mjd, y, yerr, color, marker, alpha_marker, alpha_line, line_style, line_width, label):
    """
    Plot one band's time series on `ax_b`.

    Points are sorted by MJD before plotting so that the dashed
    connector always runs left → right in chronological order.
    """
    # Sort by MJD
    order = np.argsort(mjd)
    mjd_s = mjd[order]
    y_s = y[order]
    ye_s = yerr[order]

    # Dashed line connecting points in MJD order
    ax_b.plot(mjd_s, y_s, linestyle=line_style, linewidth=line_width, color=color, alpha=alpha_line, zorder=1)

    # Markers + error bars on top
    ax_b.errorbar(
        mjd_s,
        y_s,
        yerr=ye_s,
        fmt=marker,
        color=color,
        alpha=alpha_marker,
        markersize=6 if marker == "o" else 9,
        linewidth=0,
        elinewidth=1.2,
        capsize=3,
        label=label,
        zorder=2,
    )


print("Helpers defined.")

---
## 6. Core plotting function

In [ ]:
def plot_lightcurve(
    diaobj_row,
    df_src_obj,
    df_frc_obj,
    plot_mode="flux",
    bands=None,
    colors=None,
    offsets=None,
    fig_width=FIG_WIDTH,
    fig_height=FIG_HEIGHT,
    n_date_ticks=N_DATE_TICKS,
    line_style=LINE_STYLE,
    line_alpha=LINE_ALPHA,
    line_width=LINE_WIDTH,
    spine_step=SPINE_STEP_PX,
):
    """
    Plot a multi-band light curve for a single DiaObject.

    Features
    --------
    - Wide figure (FIG_WIDTH × FIG_HEIGHT) to stretch the time axis.
    - Dashed connector lines between points, sorted by ascending MJD.
    - Independent y-axes (twinx) per band with offset spines.
    - Fixed y-axis range: MAG_MIN–MAG_MAX (mag mode) or FLUX_MIN–FLUX_MAX (flux mode).
    - Top x-axis with calendar dates.
    - Interactive zoom/pan via ipympl (when %matplotlib widget is active).

    Parameters
    ----------
    diaobj_row : pd.Series
    df_src_obj : pd.DataFrame  — DiaSources for this diaObjectId
    df_frc_obj : pd.DataFrame  — ForcedSrc for this diaObjectId
    plot_mode  : 'flux' or 'mag'
    """
    if bands is None:
        bands = BANDS
    if colors is None:
        colors = COLORS
    if offsets is None:
        offsets = OFFSETS

    diaObjectId = diaobj_row["diaObjectId"]
    ra = diaobj_row["ra"]
    dec = diaobj_row["dec"]
    n = int(diaobj_row["nDiaSources"])

    # ── Collect global MJD range ───────────────────────────────────────────────
    all_mjds = []
    if "midpointMjdTai" in df_src_obj.columns:
        all_mjds.extend(df_src_obj["midpointMjdTai"].dropna().tolist())
    if "mjd_visit" in df_frc_obj.columns:
        all_mjds.extend(df_frc_obj["mjd_visit"].dropna().tolist())

    if len(all_mjds) == 0:
        print(f"  WARNING: no MJD data for diaObjectId={diaObjectId}")
        return

    mjd_min = np.min(all_mjds) - 10
    mjd_max = np.max(all_mjds) + 10

    # ── Create figure ─────────────────────────────────────────────────────────
    fig, ax_main = plt.subplots(figsize=(fig_width, fig_height))
    ax_main.set_xlim(mjd_min, mjd_max)
    ax_main.set_xlabel("MJD (TAI)", fontsize=13)
    ax_main.yaxis.set_visible(False)  # y-axis handled by twin axes

    y_label_base = "PSF Flux (nJy)" if plot_mode == "flux" else "AB Magnitude"

    # ── Create one twin y-axis per band ───────────────────────────────────────
    ax_bands = {}
    band_has_data = {b: False for b in bands}

    for band in bands:
        ax_b = ax_main.twinx()
        offset_px = offsets[band] * spine_step
        ax_b.spines["right"].set_position(("outward", offset_px))
        ax_b.spines["right"].set_color(colors[band])
        ax_b.spines["left"].set_visible(False)
        ax_b.tick_params(axis="y", colors=colors[band], labelsize=10)
        ax_b.yaxis.label.set_color(colors[band])
        ax_b.set_ylabel(f"{band}  [{y_label_base}]", fontsize=10, labelpad=4)
        ax_bands[band] = ax_b

    # ── Plot DiaSources (marker 'o') ──────────────────────────────────────────
    for band in bands:
        df_b = df_src_obj[df_src_obj["band"] == band].copy()
        if len(df_b) == 0:
            continue

        mjd = df_b["midpointMjdTai"].values
        flux = df_b["psfFlux"].values
        ferr = df_b["psfFluxErr"].values

        if plot_mode == "mag":
            y = flux_to_mag(flux)
            yerr = flux_err_to_mag_err(flux, ferr)
        else:
            y, yerr = flux, ferr

        valid = np.isfinite(y) & np.isfinite(yerr)
        if valid.sum() == 0:
            continue

        _plot_band(
            ax_bands[band],
            mjd[valid],
            y[valid],
            yerr[valid],
            color=colors[band],
            marker="o",
            alpha_marker=0.85,
            alpha_line=line_alpha,
            line_style=line_style,
            line_width=line_width,
            label=f"{band} DiaSource",
        )
        band_has_data[band] = True

    # ── Plot ForcedSrc (marker '+') ───────────────────────────────────────────
    for band in bands:
        df_b = df_frc_obj[df_frc_obj["band"] == band].copy()
        if len(df_b) == 0:
            continue

        mjd = df_b["mjd_visit"].values
        flux = df_b["psfFlux"].values
        ferr = df_b["psfFluxErr"].values

        if plot_mode == "mag":
            y = flux_to_mag(flux)
            yerr = flux_err_to_mag_err(flux, ferr)
        else:
            y, yerr = flux, ferr

        valid = np.isfinite(y) & np.isfinite(yerr)
        if valid.sum() == 0:
            continue

        _plot_band(
            ax_bands[band],
            mjd[valid],
            y[valid],
            yerr[valid],
            color=colors[band],
            marker="+",
            alpha_marker=0.65,
            alpha_line=line_alpha * 0.6,
            line_style=line_style,
            line_width=line_width * 0.7,
            label=f"{band} Forced",
        )
        band_has_data[band] = True

    # ── Fixed y-axis limits (same range for all bands) ────────────────────────
    # In magnitude mode : MAG_MIN (bright, top) → MAG_MAX (faint, bottom)
    # In flux mode      : FLUX_MIN (faint)       → FLUX_MAX (bright)
    for band in bands:
        if plot_mode == "mag":
            # set_ylim with (bottom, top) then invert so bright is on top
            ax_bands[band].set_ylim(MAG_MAX, MAG_MIN)  # already inverted by argument order
        else:
            ax_bands[band].set_ylim(FLUX_MIN, FLUX_MAX)

    # ── Top x-axis with calendar dates ────────────────────────────────────────
    make_date_axis(ax_main, mjd_min, mjd_max, n_ticks=n_date_ticks)

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_elems = []
    for band in bands:
        if band_has_data[band]:
            legend_elems.append(
                Line2D(
                    [0],
                    [0],
                    marker="o",
                    color="w",
                    markerfacecolor=colors[band],
                    markersize=7,
                    label=f"{band} DiaSource",
                )
            )
            legend_elems.append(
                Line2D(
                    [0],
                    [0],
                    marker="+",
                    color=colors[band],
                    markersize=9,
                    linewidth=0,
                    markeredgewidth=1.5,
                    label=f"{band} Forced",
                )
            )
    # Add a legend entry for the dashed connector
    legend_elems.append(
        Line2D([0], [0], linestyle="--", color="grey", linewidth=1.2, alpha=0.7, label="MJD order connector")
    )

    ax_main.legend(
        handles=legend_elems,
        loc="upper left",
        bbox_to_anchor=(0.01, 0.99),
        ncol=3,
        framealpha=0.85,
        fontsize=10,
        title="Band  /  Source type",
        title_fontsize=11,
    )

    # ── Title ─────────────────────────────────────────────────────────────────
    mode_str = "PSF Flux (nJy)" if plot_mode == "flux" else "AB Magnitude"
    fig.suptitle(
        f"Light curve — diaObjectId = {diaObjectId}\n"
        f"RA = {ra:.2f}°   Dec = {dec:.2f}°   nDiaSources = {n}   [{mode_str}]",
        fontsize=13,
        y=1.05,
    )

    # Right margin wide enough for all spines (max offset = 5 × SPINE_STEP_PX)
    right_margin = 1.0 - (len(bands) * spine_step) / (fig_width * 96)  # 96 dpi approx
    right_margin = max(right_margin, 0.55)  # never push past 55% right edge
    plt.subplots_adjust(right=right_margin, top=0.88, bottom=0.12)
    plt.show()


print("plot_lightcurve() defined.")

---
## 7. Pre-group data by diaObjectId

In [ ]:
# Pre-group once for fast lookup inside the loop
src_grouped = df_src.groupby("diaObjectId")
frc_grouped = df_frc.groupby("diaObjectId")
print("Groupby ready.")

---
## 8. Main loop — Flux mode

In [ ]:
PLOT_MODE = "flux"  # ← change to 'mag' for AB magnitudes

print(
    f"Plotting {len(df_obj_plot)} light curves  " f"(mode={PLOT_MODE}, nDiaSources > {CUT_NDIASOURCES}) ...\n"
)

for idx, row in df_obj_plot.iterrows():
    oid = row["diaObjectId"]

    try:
        df_src_obj = src_grouped.get_group(oid).copy()
    except KeyError:
        df_src_obj = pd.DataFrame(columns=df_src.columns)

    try:
        df_frc_obj = frc_grouped.get_group(oid).copy()
    except KeyError:
        df_frc_obj = pd.DataFrame(columns=df_frc.columns)

    print(
        f"[{idx+1}/{len(df_obj_plot)}] diaObjectId={oid}  "
        f"nDiaSources={int(row['nDiaSources'])}  "
        f"DiaSrc={len(df_src_obj)}  ForcedSrc={len(df_frc_obj)}"
    )

    plot_lightcurve(
        diaobj_row=row,
        df_src_obj=df_src_obj,
        df_frc_obj=df_frc_obj,
        plot_mode=PLOT_MODE,
    )

---
## 9. Optional — AB Magnitude mode

In [ ]:
PLOT_MODE = "mag"

print(f"Re-plotting {len(df_obj_plot)} light curves in AB magnitude mode ...\n")

for idx, row in df_obj_plot.iterrows():
    oid = row["diaObjectId"]

    try:
        df_src_obj = src_grouped.get_group(oid).copy()
    except KeyError:
        df_src_obj = pd.DataFrame(columns=df_src.columns)

    try:
        df_frc_obj = frc_grouped.get_group(oid).copy()
    except KeyError:
        df_frc_obj = pd.DataFrame(columns=df_frc.columns)

    print(f"[{idx+1}/{len(df_obj_plot)}] diaObjectId={oid}")

    plot_lightcurve(
        diaobj_row=row,
        df_src_obj=df_src_obj,
        df_frc_obj=df_frc_obj,
        plot_mode="mag",
    )

---
## 10. Optional — nDiaSources distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_obj["nDiaSources"], bins=60, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(CUT_NDIASOURCES, color="red", linewidth=2, linestyle="--", label=f"Cut = {CUT_NDIASOURCES}")
ax.set_xlabel("nDiaSources", fontsize=13)
ax.set_ylabel("N DiaObjects", fontsize=13)
ax.set_title("Distribution of nDiaSources — COSMOS DDF (nmin = 200)", fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
*End of notebook `2026-03-14_DP2_DDF_DiaObjects_LightCurves.ipynb` — v3*